# 🗺️ pydeck Exercise: Bangkok Points of Interest Map
**\# From Claude**
> ## 📋 Background <br>
> You are building an interactive map of Bangkok's major landmarks using pydeck. You have a list of points of interest with their coordinates, categories, and visitor counts.


In [7]:
# Setup
import pydeck as pdk
import pandas as pd

# Data
data = pd.DataFrame({
    "name": ["Grand Palace", "Wat Arun", "Chatuchak Market",
              "Lumphini Park", "Siam Paragon", "Khao San Road"],
    "lon":  [100.4913,  100.4855,  100.5503,  100.5418,  100.5330,  100.4977],
    "lat":  [13.7500,   13.7437,   13.7999,   13.7317,   13.7459,   13.7580],
    "visitors": [20000, 15000, 35000, 10000, 40000, 25000],
    "category": ["temple", "temple", "market", "park", "mall", "nightlife"]
})

## Tasks

> ### Task 1 — ViewState
> Create a **`ViewState`** that: <br>
> -   Centers on Bangkok (`lon=100.5018, lat=13.7563`) <br>
> -   Has a zoom level of **11** <br>
> -   Has a **pitch** of **45** degrees (tilted 3D view) <br>
> -   Has a **bearing of -10** degrees <br>
> -   Restricts zoom between **8** and **16** <br>

In [8]:
view_state = pdk.ViewState(
    longitude=100.5018,
    latitude=13.7563,
    zoom=11,
    min_zoom=8,
    max_zoom=16,
    pitch=45,
    bearing=-10
)

> #### Key points: <br>
> - `longitude` / `latitude` set the map center
> - `zoom=11` is city-level (0 = whole world, 24 = building detail)
> - `pitch=45` tilts the map for a 3D perspective
> - `bearing=-10` rotates slightly left of true north
> - `min_zoom` / `max_zoom` restrict how far users can zoom

> ### Task 2 — ScatterplotLayer
> Create a `ScatterplotLayer` named `"poi-layer"` that: <br>
> - Uses the `data` DataFrame above <br>
> - Reads position from `lon` and `lat` columns <br>
> - Sets `get_radius` to scale with visitors: use the expression `"visitors / 20"` <br>
> - Sets `get_fill_color` to `[255, 140, 0, 180]` (orange, semi-transparent) <br>
> - Sets `get_line_color` to `[0, 0, 0]` <br>
> - Enables `pickable=True` and `stroked=True` <br>

In [9]:
scatter_layer = pdk.Layer(
    "ScatterplotLayer",          # type (required, deck.gl class name)
    data=data,                   # pandas DataFrame
    id="poi-layer",              # unique layer name
    get_position=["lon", "lat"], # expression parser reads column names
    get_radius="visitors / 20",  # JS expression: scales radius by visitors
    get_fill_color=[255, 140, 0, 180],  # RGBA: orange, semi-transparent
    get_line_color=[0, 0, 0],    # black outline
    pickable=True,               # enables hover/click events
    stroked=True,                # draws the outline
)

> #### Key points: <br>
> - `type` is the first positional arg — use the exact deck.gl class name
> - `get_position` uses the expression parser to read `lon` and `lat` columns
> - `"visitors / 20"` is a JS expression string evaluated per row by deck.gl
> - Colors are `[R, G, B, A]` where A is 0–255 (not 0–1)
> - `snake_case` kwargs in pydeck = `camelCase` in deck.gl docs (e.g. `getFillColor` → `get_fill_color`)

> ### Task 3 — Deck
> Assemble a `pdk.Deck` that: <br>
> - Includes your layer
> - Uses your `ViewState` as `initial_view_state`
> - Sets `map_style` to `pdk.map_styles.LIGHT` instead of `"mapbox://styles/mapbox/light-v9"` because mapbox need token
> - Sets `map_style` to 
> - Adds a tooltip showing `"name"` and `"visitors"` fields like:
> - `{"html": "<b>{name}</b><br>Visitors: {visitors}"}`
> - Exports the result to `"bangkok_poi.html"`

In [ ]:
# r stand for renderer
r = pdk.Deck(
    layers=[scatter_layer],
    initial_view_state=view_state,
    # map_style="mapbox://styles/mapbox/light-v9", # need token
    map_style=pdk.map_styles.LIGHT,
    tooltip={"html": "<b>{name}</b><br>Visitors: {visitors}"}
)


# r.to_html("bangkok_poi.html") # export to HTML file
r.show()

> #### Key points: <br>
> - `layers` takes a list — you can add multiple layers
> - `initial_view_state` takes your `pdk.ViewState` object
> - `tooltip` dict uses `{column_name}` placeholders from your data
> - `to_html()` exports a self-contained HTML file you can open in a browser

> ### Bonus Task
> Add a **second layer** — a `TextLayer` — that labels each point with its `name`. Set `get_text="name"`, `get_size=14`, `get_color=[50, 50, 50]`, and `get_position=["lon", "lat"]`.

In [11]:
text_layer = pdk.Layer(
    "TextLayer",
    data=data,
    get_position=["lon", "lat"],
    get_text="name",             # column name as string (expression parser)
    get_size=14,
    get_color=[50, 50, 50],      # dark gray text
    get_alignment_baseline="'bottom'",  # string constant (note extra quotes)
)

r = pdk.Deck(
    layers=[scatter_layer, text_layer],   # both layers in the list
    initial_view_state=view_state,
    # map_style="mapbox://styles/mapbox/light-v9", # need token
    map_style=pdk.map_styles.DARK,
    tooltip={"html": "<b>{name}</b><br>Visitors: {visitors}"}
)

# r.to_html("bangkok_poi.html")
r.show()

> #### Key points:
> - Multiple layers go in the `layers=[]` list; they render in order
> - `get_text="name"` — the expression parser reads `name` as a column
> - String constants like `'bottom'` need double wrapping:` "'bottom'"` — outer quotes are Python, inner quotes tell deck.gl it's a literal string, not a column name (use `pdk.types.String("bottom")` as the cleaner alternative)

## 🔑 Concepts Summary

| Concept | What it does |
|---|---|
| `pdk.Layer(type, data, **kwargs)` | Defines a visualization layer |
| `pdk.ViewState(lat, lon, zoom, pitch, bearing)` | Sets camera position & angle |
| `pdk.Deck(layers, initial_view_state, tooltip)` | Assembles the full map |
| `get_position=["lon", "lat"]` | Expression parser reads column names |
| `"visitors / 5"` (string) | JS expression evaluated per row |
| `[R, G, B, A]` | Color as list, A is 0–255 |
| `r.to_html("file.html")` | Exports standalone HTML map |
| `r` or `r.show()` | Renders map inline in Jupyter notebook |